# 01 シナリオと想定QA（ハンズオン1: Genie Agent）

ローコードの **Genie Agent** で、品質保証部の分析エージェントを作ります。修理受付のデータ（テーブル）と社内文書（PDF・PPTX）を横断して、**不具合の原因と対策まで答えるエージェント（RAG）を、コードを書かずに作れること**を確かめます。

このノートブックは読むためのものです。最後の「付録B 答え合わせ用の SQL」だけ、実行して数値を確かめられます。

| # | 内容 |
|---|---|
| 1 | 想定シナリオ |
| 2 | 使うデータ |
| 3 | ハンズオンの流れと事前準備 |
| 4 | Step 2: データを確認する |
| 5 | Step 3: Genie Agent を作る（**手動**: テーブルと Volume の追加、指示の入力） |
| 6 | Step 4: Agent モードで分析する（質問1〜3の想定回答と検証ポイント） |
| 7 | Step 5: 共有する |
| 8 | まとめ |
| 付録 | A. データに仕込んだ筋書き（講師向け） ／ B. 答え合わせ用の SQL |

## 1. 想定シナリオ

### 会社
家電メーカー「**ブリックス電機**」（架空）。炊飯器・電気ケトル・電子レンジ・掃除機などの生活家電を製造し、全国の家電量販店と EC で販売しています。修理の受付はコールセンター（オペレーター約50人）が電話と Web で行い、出張修理・持込修理・製品交換を手配しています。

### 部門: 品質保証部（数人のチーム）
| | |
|---|---|
| **業務** | 修理の傾向を監視し、不具合の原因を調べて、対策（無償点検・設計変更）を決める |
| **課題** | ・修理受付のデータと、会議資料・技術情報などの文書がバラバラで、原因の特定に時間がかかる<br>・分析はデータ担当に頼むため、待ち時間も長い |
| **実現したいこと** | 「修理データと社内文書を横断して、不具合の原因を自分たちで素早く突き止め、対策を決めたい」 |

### このハンズオンで作るもの
| | |
|---|---|
| 作るもの | ドメインに特化した、チームで使うエージェント（品質保証部の分析エージェント） |
| 作り方 | Genie Agent（ローコード）。Agent モードで、SQL の集計と文書の検索を組み合わせて複数のステップで分析する |
| データ | 構造化データ（テーブル4つ）＋非構造化データ（PDF・PPTX 34ファイル） |
| 伝えたいこと | **ローコードでも本格的なエージェントが作れる**（根拠の SQL と文書を示すので、答えを確かめられる） |

## 2. 使うデータ

対象期間は **2025年4月〜2026年9月（18か月）** です。データの最新月は2026年9月です。

### 2-1. テーブル（Unity Catalog、列コメント付き）

| テーブル | 行数 | 内容 | 主な列 |
|---|---|---|---|
| `products` | 10 | 製品 | `model_no`（型番）, `product_name`, `category`, `warranty_months`（保証期間） |
| `production_lots` | 783 | 製造ロット | `lot_no`, `model_no`, `factory`（新潟・タイ・蘇州）, `production_month`, `quantity`（製造台数） |
| `sales_monthly` | 1,080 | 月次の販売台数（型番×地域） | `sales_month`, `model_no`, `region`, `units_sold` |
| `repair_requests` | 11,884 | 修理受付 | `request_id`, `received_at`, `channel`（電話・Web）, `model_no`, `serial_no`, `lot_no`, `purchase_date`, `symptom_category`, `result_category`, `warranty_status`, `service_cost_jpy`, `operator_memo` |

**テーブルのつながり**（主キー・外部キーを設定済み）

```
products (model_no) ─┬─< production_lots (lot_no, model_no) ─< repair_requests (lot_no)
                     ├─< sales_monthly   (model_no)
                     └─< repair_requests (model_no)
```

- 型番（`model_no`）とロット番号（`lot_no`）で、製品・製造・販売・修理受付がつながります。
- ロット番号は「型番の略号-製造年月+ロット記号」です（例: `RC5-2604B` = RC-500 を2026年4月に製造した B ロット）。製造番号（`serial_no`）の先頭9文字がロット番号です。
- `result_category`（対応結果）: 持込修理／出張修理／製品交換／**修理不要**（電話・メールの案内で解決）／**異常なし**（修理を手配したが、点検の結果、故障ではなかった）
- ⚠️ `operator_memo`（対応メモ）には、お客様の氏名・電話番号・住所がそのまま書き込まれています（ダミーデータ）。

**販売台数あたりの受付率**（販売1,000台あたり）で比べると、季節で販売台数が大きく変わる製品（扇風機・ヒーター・加湿器）も公平に比べられます。

### 2-2. 文書（Volume `docs`）

| フォルダ | 文書 | 形式 | 数 | 内容 |
|---|---|---|---|---|
| `manuals/` | 取扱説明書 | PDF | 10（製品ごとに1本） | 使いかた、お手入れ、トラブル対処の表、エラー表示の一覧 |
| `quality_meetings/` | 品質会議の資料 | PPTX | 18（月1回×18回） | 品質保証部の月次会議。修理受付の概況、部品・工程の変更（変更点管理）、対策の進捗、決定事項 |
| `service_bulletins/` | サービス技術情報 | PDF | 5 | 修理担当者・コールセンター向けの通達。既知の不具合と対処 |
| `policies/` | 保証規定・修理料金表 | PDF | 1 | 保証の条件、修理料金（技術料・部品代・送料）の目安 |

社内の品質会議の資料は、このハンズオン1の分析にだけ使います（お客様への案内の根拠にはしません）。

## 3. ハンズオンの流れと事前準備

| Step | 内容 | 時間 |
|---|---|---|
| 1 | 説明（会社・業務・課題、テーブルと文書の紹介） | 15分 |
| 2 | データの確認（Catalog Explorer でテーブルと列コメント、Volume の PDF・PPTX を見る） | 15分 |
| 3 | 作成（Genie Agent を作る、テーブルと文書（Volume）を接続、業務の説明（指示）を書く） | 20分 |
| 4 | 分析（Agent モードで質問1〜3を順に投げ、答えと根拠を確かめる） | 40分 |
| 5 | 共有（権限を付けて共有する、まとめ） | 10分 |

### 事前準備（講師・ワークスペース管理者）
| 項目 | 内容 |
|---|---|
| プレビューの有効化（**管理者**） | ワークスペースの［プレビュー］で「**Genie エージェントによるボリューム内のファイル分析**（Analyze Files in Volumes with Genie Agents）」をオンにする（ベータ） |
| パートナーが提供する AI 機能 | アカウントとワークスペースの両方で有効にする（Genie を使うための前提） |
| データの準備 | `00_セットアップ` を実行する（スキーマ・Volume・テーブルを作り、CSV と文書をコピーする） |
| SQL ウェアハウス | Pro またはサーバーレスの SQL ウェアハウス。参加者に `CAN USE` |
| 参加者の権限 | テーブルへの `SELECT`、`docs` Volume への `READ VOLUME`（Genie Agent は質問した人の権限でデータを読む） |
| 対応リージョン | 文書の分析（コンテンツ検索）は対応リージョンのみ。AWS の米国東部（バージニア北部）・東京などは Geo 間処理なしで使える |

## 4. Step 2: データを確認する（Catalog Explorer）

1. 左のサイドバーの［**カタログ**］を開き、`00_セットアップ` で指定したカタログ → スキーマ（既定は `bricks_electronics`）を開きます。
2. テーブル `repair_requests` を開き、［**概要**］で列コメントを、［**サンプルデータ**］で中身を見ます。
   - `operator_memo` に個人情報が入っていることを確かめます。
   - `lot_no` と `production_lots` の関係（ロット番号の付け方）を確かめます。
3. Volume `docs` を開き、`manuals/`・`quality_meetings/`・`service_bulletins/`・`policies/` の PDF・PPTX をプレビューします。
   - 例: `quality_meetings/quality_meeting_2026-09.pptx`（最新の品質会議）、`manuals/manual_RC-500.pdf`

> 💡 列コメントとファイル名は、Genie Agent が「どのデータを使えばよいか」を判断する手がかりになります。ここでは、どんな情報がどこにあるかを把握しておきましょう。

## 5. Step 3: Genie Agent を作る（手動）

### 5-1. Genie Agent を作り、テーブルを追加する
1. 左のサイドバーの［**Genie エージェント**］をクリックし、右上の［**新規**］をクリックします。
2. データソースとして、次の **4つのテーブル** を選び、［**作成**］をクリックします。
   - `products` / `production_lots` / `sales_monthly` / `repair_requests`
3. 作成すると Genie Code が起動し、テーブルの説明などを提案します。内容を確認して、採用するか閉じます。
4. ［**構成**］→［**設定**］で、タイトル（例: `ブリックス電機 品質保証部 分析エージェント`）と、SQL ウェアハウスを設定します。

### 5-2. 文書（Volume）を追加する
1. ［**構成**］→［**データ**］（ソース）を開き、［**追加**］をクリックします。
2. Volume `docs` を選びます（Volume 全体が対象になります。フォルダやファイルの一部だけを追加することはできません）。
3. 説明の欄に、次の文章を貼り付けます。Agent はこの説明を手がかりに、関係する文書を探します。

```
ブリックス電機の社内文書（PDF・PPTX、34ファイル）。
- manuals/manual_<型番>.pdf: 製品ごとの取扱説明書（10本）。使いかた、お手入れ、「故障かな？と思ったら」（トラブル対処の表）、エラー表示の一覧、保証期間
- quality_meetings/quality_meeting_<年>-<月>.pptx: 品質保証部の月次の品質会議の資料（2025年4月度〜2026年9月度の18本）。修理受付の概況、部品・工程の変更（変更点管理）、対策の進捗、決定事項
- service_bulletins/STI-<番号>_<型番>_<件名>.pdf: サービス技術情報（5本）。既知の不具合（対象ロット・原因・対策）と、受付時の対応の手順
- policies/warranty_and_repair_fees.pdf: 保証規定・修理料金表。保証の条件、技術料・部品代・送料・点検料
品質会議の資料は社内向けで、原因の調査や対策の検討に使う。
```

> 💡 Volume を追加すると、**コンテンツ検索**が自動で有効になり、ファイルが解析（`ai_parse_document`）されてインデックスが作られます。最初は数分かかることがあります。ファイルを追加・変更・削除すると、インデックスは自動で同期されます。
> ⚠️ `raw_data` Volume（CSV）は追加しません。関係のないファイルは、文書の検索の精度を下げます。

### 5-3. 業務の説明（指示）を書く
［**構成**］→［**指示**］を開き、次の文章を貼り付けます。データの意味（受付率の定義、「この半年」の範囲、個人情報の扱い）を、ここでエージェントに教えます。

```
# あなたの役割
あなたは、家電メーカー「ブリックス電機」の品質保証部の分析エージェントです。修理受付のデータ（テーブル）と社内文書（Volume）を組み合わせて、修理の傾向の監視、不具合の原因の調査、対策（無償点検・設計変更など）とその費用の検討を手伝います。

# データの使い分け
- 件数・率・金額は、必ずテーブルを集計して答える。
- 原因や背景（部品・工程の変更、過去の不具合と対策、使いかたの説明、費用の単価）は、文書（docs Volume）から探す。
  - quality_meetings/: 品質保証部の月次の品質会議の資料。部品・工程の変更（変更点管理）、対策の進捗、決定事項
  - service_bulletins/: サービス技術情報。既知の不具合と、受付時・修理時の対応
  - manuals/: 製品ごとの取扱説明書。使いかた、お手入れ、トラブル対処の表、エラー表示の一覧
  - policies/: 保証規定・修理料金表。保証の条件、技術料・部品代・送料
- 答えには、根拠にしたテーブル（集計の条件）と文書の名前を示す。

# 指標の定義
- 受付率 = 修理受付の件数 ÷ 販売台数 × 1,000（販売1,000台あたり）。製品の比較や増減の判断は、件数ではなく受付率で行う（扇風機・ヒーター・加湿器は季節で販売台数が大きく変わるため）。sales_monthly は地域別なので、型番と月で合計してから割る。
- 「この半年」は、データの最新月（2026年9月）までの6か月（2026年4月〜9月）。比べる相手は、その前の6か月（2025年10月〜2026年3月）か、前年の同じ時期。
- 「実際には修理が不要だったもの」は、result_category が「修理不要」（電話・メールの案内で解決）と「異常なし」（修理を手配したが、点検の結果、故障ではなかった）の受付。
- ロットごとの不具合の多さは、ロットごとの受付件数 ÷ production_lots.quantity（製造台数）で比べる。
- 費用の目安は、修理料金表の単価（技術料・部品代・送料）と、過去の無償交換の実績（実施率・1台あたりの費用・告知費用）を根拠にする。

# 注意
- operator_memo には、お客様の氏名・電話番号・メールアドレス・住所が含まれる。回答に個人情報を出さず、件数や傾向に要約する。
- 型番が似ている製品（RC-500 と RC-550）を取り違えない。
```

### 5-4. （任意）サンプルの質問
［**構成**］→［**設定**］のサンプルの質問に、次の3つを入れておくと、参加者がクリックだけで質問できます。
- この半年で修理の受付が増えている製品を特定し、原因を分析して
- 影響の範囲と、取るべき対策、その費用の目安を出して
- 修理受付のうち、実際には修理が不要だったものの傾向を分析して、改善策を提案して

## 6. Step 4: Agent モードで分析する

**文書についての質問は Agent モードでしか動きません。** チャット欄で **Agent モードをオン** にしてから質問します。

- Agent は、SQL の集計と文書の検索を組み合わせ、複数のステップで分析します（1つの質問に1〜3分ほどかかります）。
- 回答の番号（`[1]` など）から、**根拠の SQL** と **引用した文書（ページ）** を確かめられます。
- 質問1→2→3 は **同じ会話の中で** 続けて投げます（質問2は、質問1の結論を前提にしています）。

> 集計を頼むのではなく、**原因と対策まで考えさせる問い**にしているのがポイントです。見るポイントは、Agent がどのテーブルを集計し、どの文書を読んで結論を出したかです。

想定回答の数値は、`00_セットアップ` で作ったデータから計算した値です（付録B の SQL で確かめられます）。Agent の言い回しや、表の出し方は毎回変わります。

### 質問1「この半年で修理の受付が増えている製品を特定し、原因を分析して」

**ねらい**: 件数ではなく受付率で比べられるか（季節製品のひっかけ）。データで「どこで」起きているかを絞り込み、文書で「なぜ」を突き止められるか。

**Agent の動き（想定）**
1. 製品ごとに、この半年（2026年4〜9月）と前の半年・前年同期の受付率を比べる（`repair_requests` × `sales_monthly`）
2. RC-500 の月別の推移と症状を見る
3. RC-500 をロット・工場・製造月で分ける（`production_lots` と結合し、ロットの製造台数で割る）
4. 文書で原因を探す: 品質会議の資料の「部品・工程の変更」と「対策の進捗」、RC-500 の取扱説明書

**想定回答**

| 型番 | 2025/10〜2026/3 件数 | 受付率 | 2026/4〜9 件数 | 受付率 | 受付率の増減 |
|---|---:|---:|---:|---:|---:|
| **RC-500（IH炊飯器）** | 557 | 9.4 | **1,162** | **19.2** | **+103%** |
| FN-210（扇風機） | 29 | 9.4 | 543 | 8.9 | −5%（件数は約19倍） |
| VC-700（掃除機） | 682 | 12.9 | 591 | 11.6 | −10% |
| KT-200（電気ケトル） | 757 | 9.4 | 478 | 6.7 | −29% |
| そのほか6製品 | | | | | ±5%前後で横ばい |

- **受付が増えているのは RC-500**。受付率は2026年4月 9.9 → 5月 10.2 → 6月 13.1 → 7月 20.4 → 8月 29.2 → 9月 **34.3** と加速している。症状は「**途中で止まる**」が中心（月13件 → 216件）。
- 増えているのは **タイ工場で2026年4〜6月に製造した6ロット**（`RC5-2604A/B`・`RC5-2605A/B`・`RC5-2606A/B`）。製造1,000台あたりの受付は 28〜64件で、ほかのロット（多くて12件程度。2026年製造分は7件以下）の数倍。新潟工場の同じ月のロットや、2026年7月以降のロットは平常。
- 文書から分かる原因:
  - `quality_meeting_2026-03.pptx`（変更点管理）: タイ工場で **温度ヒューズの調達先を A社 から K社 に変更（2026年4月製造分から）**。コスト低減のため。評価は n=30 で合格していた。
  - `quality_meeting_2026-06.pptx`: K社品の納入遅延のため、**2026年7月製造分から A社 の品に戻した**（品質の理由ではない）。
  - `quality_meeting_2026-07〜09.pptx`: 受付の増加を監視。延長コードを想定した案内では改善しない。発煙・発火の報告はない。9月度の中間報告で、**回収品12台のうち9台で温度ヒューズが溶断し、定格（145℃）より低い温度で作動するものがあった**。
  - `manual_RC-500.pdf`: 延長コードを使っていないのに炊飯の途中で電源が切れるときは、内部の安全装置（温度ヒューズなど）が作動した可能性がある。
- **結論**: K社製の温度ヒューズを使った期間（タイ工場の2026年4〜6月製造分）の製品で、温度ヒューズが本来より低い温度で作動し、炊飯の途中で電源が切れている可能性が高い。

**検証ポイント**
- ☐ 件数ではなく **受付率（販売1,000台あたり）** で比べているか（`sales_monthly` を型番×月で合計してから割っているか）
- ☐ **FN-210 を「増えている製品」にしていないか**（件数は夏に約19倍だが、受付率は横ばい＝季節要因）
- ☐ ロットの製造台数（`production_lots.quantity`）で割って、**タイ工場の2026年4〜6月製造分**に絞り込めているか
- ☐ 品質会議の資料の **2026年3月度（変更）と 6月度（元に戻す）** を見つけ、ロットの範囲と結び付けているか
- ☐ 根拠の SQL と、引用した文書（ファイル名・ページ）が示されているか
- ☐ 回答に、お客様の氏名・電話番号などの個人情報が出ていないか

**比べてみよう: 文書（Volume）がない場合**

テーブルだけの Genie Agent で同じ質問をすると、RC-500 の増加と、タイ工場の2026年4〜6月製造ロットへの集中までは、SQL だけで突き止めます。しかし「その時期のタイ工場での部品または工程の変更が原因の可能性が高いが、**文書を確認できないため、ここまでしか言えない**」という答えで止まります。**テーブルで「何が・どこで」、文書で「なぜ」**が分かります。

**深掘りの質問例**
- 「RC-500 の『途中で止まる』を、工場と製造月で分けて、製造1,000台あたりで見せて」
- 「2026年に、部品や工程の変更は他にもあった？ それぞれ受付への影響はあった？」
- 「VC-700 は前年より受付率が下がっている。何があった？」（2025年のバッテリー不良と無償交換）

### 質問2「影響の範囲と、取るべき対策、その費用の目安を出して」

**ねらい**: 質問1の結論を前提に、影響する台数を数え、規定と過去の実績（文書）を根拠に、対策と費用を見積もれるか。

**Agent の動き（想定）**
1. 対象ロットの製造台数と、これまでの受付件数を集計する（`production_lots`・`repair_requests`）
2. 修理料金表から、1台あたりの費用（技術料・部品代・送料）を調べる（`warranty_and_repair_fees.pdf`）
3. 保証規定で、保証期間外でも無償にできるかを確かめる
4. 過去の無償交換の実績（実施率・費用・告知費用）を品質会議の資料から探す（VC-700 の例）

**想定回答**

| 項目 | 内容 |
|---|---|
| 対象 | RC-500、タイ工場の2026年4〜6月製造の **6ロット**（`RC5-2604A/B`・`RC5-2605A/B`・`RC5-2606A/B`） |
| 対象台数 | **12,650台**（1,900＋2,120＋2,350＋2,440＋1,720＋2,120） |
| これまでの受付 | **605件**（製造台数の約4.8%）。うち「途中で止まる」486件・「電源が入らない」94件。すべて保証期間内。地域は全国（関東228件、近畿100件、中部91件…）。当社負担は約478万円 |
| 今後 | 6月製造ロットは購入からの日数が浅く、受付は **まだ増える見込み**。2026年7月製造分からは A社 の温度ヒューズに戻しており、範囲は6ロットで止まる |

**取るべき対策**
1. **対象6ロットの無償点検・温度ヒューズの交換**。保証規定 第6条（技術情報で告知した対象製品は、保証期間にかかわらず無償で点検・修理）に基づく。
2. **サービス技術情報の発行**（コールセンターが製造番号で対象を判別し、使用の中止をお願いして、引き取りを手配する）。※ハンズオン2の冒頭で、この技術情報を追加します。
3. **お客様への告知**（DM・Web・販売店）。
4. **再発の防止**: K社 の温度ヒューズの採用をやめる（7月製造分から元に戻し済み）。調達先を変えるときの評価（今回は n=30 で合格）と、受入検査（作動温度の確認）を見直す。
5. 2026年7月以降に製造したロットの受付率を、引き続き監視する。

**費用の目安**

| 内訳 | 金額 | 根拠 |
|---|---:|---|
| 1台あたりの費用 | **7,040円** | 修理料金表: 技術料（炊飯器）4,400円＋温度ヒューズ 660円＋引き取りの往復送料（中型）1,980円。データでも、対象ロットの持込修理の当社負担の平均は約7,050円 |
| 実施率 | 62.6% | `quality_meeting_2026-06.pptx`: VC-700 の無償交換の最終実績（実施 6,881台／対象 10,990台）。この実績を、今後の見積もりの基準にすると決めている |
| 点検・交換の費用 | **約5,600万円** | 12,650台 × 62.6% × 7,040円 ≒ 5,575万円 |
| 告知費用 | 約420万円 | VC-700 の無償交換の告知費用（`quality_meeting_2025-06.pptx`・`2026-06.pptx`） |
| **合計の目安** | **約6,000万円** | 全数（12,650台）を交換した場合の上限は、約8,900万円＋告知費用 |

**検証ポイント**
- ☐ 対象を **6ロット・12,650台** と、ロットの製造台数から数えているか（RC-500 全体にしていないか）
- ☐ 1台あたりの費用を **修理料金表（PDF）の単価** から組み立てているか
- ☐ **実施率と告知費用を、過去の実績（VC-700 の無償交換）** から引いてきているか
- ☐ 保証期間外の製品も無償にする根拠（保証規定 第6条）に触れているか
- ☐ 前提（実施率など）を明示して、幅のある見積もりにしているか

**比べてみよう: 文書（Volume）がない場合**

テーブルだけでは、対象ロットと台数（12,650台）や、これまでの費用（約477万円）は出せます。しかし、修理料金表も、過去の無償交換の実施率・告知費用も読めないため、「同じ不具合率が続けば、追加で約540件・約440万円」という、**受付を待つだけの見積もり**になります。無償点検を前提とした費用（約6,000万円）は、文書があって初めて出せます。

**深掘りの質問例**
- 「実施率を50%・70%・100%にしたときの費用を表にして」
- 「過去の無償交換（VC-700）は、どんな経緯で決めて、どれくらいの費用がかかった？」
- 「対象ロットの受付は、地域ごとにどれくらい？ 告知の優先順位は？」

### 質問3「修理受付のうち、実際には修理が不要だったものの傾向を分析して、改善策を提案して」

**ねらい**: 「修理が不要だった」をデータの言葉（`修理不要`＋`異常なし`）に置き換えられるか。型番×症状・チャネルで傾向をつかみ、取扱説明書と技術情報から、なぜ防げていないかと改善策を導けるか。

**Agent の動き（想定）**
1. `result_category` が「修理不要」「異常なし」の受付を、型番×症状・チャネル（電話・Web）・月で集計する
2. 「異常なし」の当社負担の費用（送料・点検料）を集計する
3. 取扱説明書の「故障かな？と思ったら」と、サービス技術情報（STI-2025-03・STI-2026-01・STI-2026-02）を読み、一次対処で解決できる内容か、どう運用されているかを確かめる

**想定回答**

| 対応結果 | 件数 | 割合 |
|---|---:|---:|
| 修理不要（電話・メールの案内で解決） | 3,394 | 28.6% |
| 異常なし（修理を手配したが、点検の結果、故障ではなかった） | 1,605 | 13.5% |
| **合計（実際には修理が不要だったもの）** | **4,999** | **42.1%** |

- **チャネルの差が大きい**: 「異常なし」の割合は、電話 5.8% に対して **Web 24.4%**。「異常なし」1,605件のうち 1,204件（75%）が Web 受付。電話ではオペレーターが一次対処を案内して「修理不要」で終わる内容が、Web では **そのまま引き取りの手配**になっている。
- **多い型番×症状**（その症状の受付のうち、修理が不要だった割合）:
  - AP-400 におい 85%（脱臭フィルターの汚れ・交換時期）、KT-200 におい 86%（使い始めのにおい）・異音 91%（沸騰の音）
  - RC-500 性能低下 79%（水量の目盛り・炊飯メニュー）・異音 84%（IH の動作音）、RC-550 性能低下 86%
  - RC-500・RC-550 エラー表示 62%（E1＝内釜のセット、RC-550 の U1・U2＝内ぶた・蒸気口キャップの付け忘れ）
  - MW-300 性能低下 67%（出力の設定・容器）、VC-700 性能低下 40%（フィルターの目詰まり）
- どれも **取扱説明書の「故障かな？と思ったら」に処置が書いてある** 内容。
- **技術情報の効果は電話だけ**: AP-400 のにおい（STI-2025-03、2025年12月）の後、電話の「異常なし」率は 9%→5% に下がったが、Web は 47%→44% でほぼ変わらない。RC-550 の U1・U2（STI-2026-02、2026年4月）も、電話 8%→3% に対して Web は 38%→45%。技術情報には「**Web 受付は、現行の運用ではそのまま引き取りを手配している**」とある。
- **費用**: 「異常なし」の当社負担（往復送料＋点検料）は18か月で **約549万円**（年 約370万円、1件 約3,400円）。ほかに、修理不要の対応にかかるオペレーターの工数がある。
- 補足: 2026年7〜9月は「修理が不要だった」割合が下がって見えるが、件数はほぼ横ばいで、RC-500 の故障の増加で受付全体が増えたためである。

**改善策（例）**
1. **Web の受付フォームにセルフチェックを入れる**: 型番と症状（エラー表示）を選ぶと、取扱説明書のトラブル対処を表示し、解決しない場合だけ引き取りを申し込める。
2. **Web 受付も、手配の前に一次対処のメールを返す**運用にする（技術情報の手順を、Web の返信にも適用する）。
3. **技術情報の一次対処の手順を、ほかの型番×症状にも広げる**（RC-500 の炊き上がり、KT-200 のにおい・異音、MW-300 の温まらない、VC-700 の吸引力）。
4. **製品と取扱説明書の改善**: 本体の表示で U1・U2・E1 の意味が分かるようにする、フィルターの交換時期を知らせる、など。
5. 「異常なし」の割合（チャネル別）を、品質会議の指標に加えて毎月確認する。

**検証ポイント**
- ☐ 「修理不要」と「異常なし」の **両方** を対象にしているか（どちらか一方だけになっていないか）
- ☐ **チャネル（電話・Web）** の違いに気づいているか
- ☐ 型番×症状で、**その症状の受付に占める割合** を見ているか（件数だけだと、販売台数の多い製品が上位に来る）
- ☐ 取扱説明書・技術情報を引用して、「一次対処で解決できる内容」であることを示しているか
- ☐ 費用（`service_cost_jpy`）で、改善の効果の大きさを示しているか
- ☐ **期間を取り違えていないか**（「異常なし」の約549万円は18か月分。年間なら約370万円）

**比べてみよう: 文書（Volume）がない場合**

テーブルだけでも、「修理不要」「異常なし」の割合、Web で「異常なし」が多いこと、多い型番×症状、費用までは集計でき、「Web の受付に事前診断を入れる」といった改善策も出てきます。ただし、**取扱説明書に処置が書いてある内容であること**や、**技術情報を出した後も Web だけ減っていない理由**（Web は現行の運用ではそのまま引き取りを手配している）は、文書がないと分かりません。テーブルだけの Agent は「具体的な案内の文言や診断の基準は、取扱説明書や技術情報を踏まえて設計するのが望ましい」と、文書の確認を求めて終わります。

**深掘りの質問例**
- 「技術情報（STI）を出す前と後で、電話と Web の『異常なし』の割合はどう変わった？」
- 「Web 受付の『異常なし』をなくせたら、年間でいくら減らせる？」
- 「RC-550 の U1・U2 の受付を、取扱説明書のどこを案内すれば解決できる？」

## 7. Step 5: 共有する

Genie Agent は、**質問した人の権限**でテーブルと Volume を読みます。チームのメンバーに共有するときは、次の両方が必要です。

| 対象 | 付ける権限 | 付ける場所 |
|---|---|---|
| Genie Agent | `CAN RUN`（使うだけ）/ `CAN EDIT`（指示も直す） | Genie Agent の右上の［**共有**］ |
| テーブル4つ | `SELECT`（＋カタログの `USE CATALOG`、スキーマの `USE SCHEMA`） | Catalog Explorer の［権限］ |
| `docs` Volume | `READ VOLUME` | Catalog Explorer の［権限］ |
| SQL ウェアハウス | Genie Agent に設定したウェアハウスの資格情報で実行される（作成者の資格情報が埋め込まれる） | — |

- `READ VOLUME` がない人が文書について質問すると、エラーになります。
- ユーザーには、ワークスペースへのアクセスと Databricks SQL へのアクセスの両方の資格が必要です（ベータの間、アカウントだけのユーザーは文書の分析を使えません）。
- テーブルと文書を **同じ Unity Catalog の権限の仕組み** で守れることが、ローコードでもチームに安心して広げられる理由です。

## 8. まとめ

| 観点 | このハンズオンで確かめたこと |
|---|---|
| ローコード | テーブルと Volume を選び、業務の説明（指示）を書くだけで、品質保証部の分析エージェントができた |
| 本格的なエージェント（RAG） | Agent モードが、SQL の集計 → ロットの絞り込み → 文書の検索 → 原因と対策・費用の提案、と複数のステップで分析した |
| 構造化＋非構造化 | テーブルで「何が・どこで」、文書で「なぜ・どうする」。片方だけでは結論に届かない |
| 確かめられる答え | 根拠の SQL と、引用した文書（ページ）を示すので、人が確認して判断できる |
| ガバナンス | テーブルも文書も Unity Catalog で管理し、質問した人の権限で読む |

次のハンズオン2では、同じ会社のデータで、コールセンター約50人が毎日使う **回答支援エージェント** をハイコードで作り、評価・管理・デプロイ・監視（AgentOps）までを Databricks の中で行います。

## 付録A. データに仕込んだ筋書き（講師向け）

| 出来事 | データ | 文書 |
|---|---|---|
| **RC-500 温度ヒューズ**（この半年の主役） | タイ工場の2026年4〜6月製造の6ロット（12,650台）で「途中で止まる」が増える。2026年7月製造分からは平常 | 品質会議 2026年3月度（K社に変更）、6月度（A社に戻す）、7〜9月度（監視・解析の中間報告）、取扱説明書（安全装置）。**サービス技術情報はまだない**（ハンズオン2で追加） |
| VC-700 バッテリー（2025年） | タイ工場の2025年2〜3月製造の4ロット（10,990台）で、2025年5〜8月に受付率が 20〜27 に上がり、その後収束 | STI-2025-01（無償交換）、品質会議 2025年4〜6月度（原因・決定・費用見込み）、各月の進捗、2026年6月度（最終実績: 実施率62.6%、告知費用 約420万円） |
| KT-200 ふたのロックつまみ（2025年秋） | 蘇州工場の2025年7〜8月製造の4ロットで「破損」が増え、収束 | 品質会議 2025年7月度（成形条件の変更）、9月度（元に戻す）、STI-2025-02 |
| 季節製品のひっかけ | FN-210（夏）・HT-310／HM-220（冬）は件数が大きく動くが、受付率は横ばい | 品質会議 2025年4月度（受付率を標準の指標にする） |
| 修理が不要だった受付 | 全体の42%。Web 受付で「異常なし」が多い | 取扱説明書（トラブル対処の表）、STI-2025-03・2026-01・2026-02（電話の一次対処。Web は現行の運用ではそのまま手配） |
| スライドのサンプル行 | `R-2607-09140`・`09141`・`09146`・`09150` は、2026年7月1日の最初の4件 | — |

- 受付番号（`request_id` の末尾5桁）は、修理以外の問い合わせと共通で毎年1月に00001から振り直す連番のため、修理受付だけを見ると飛び番になります。
- 個人情報（氏名・電話番号・メールアドレス・住所）はすべて架空です。メールアドレスは `example.com` などの例示用ドメインです。

## 付録B. 答え合わせ用の SQL

想定回答の数値を確かめるための SQL です。上のウィジェットで、`00_セットアップ` と同じカタログ名・スキーマ名を指定してから実行します（サーバーレスまたは SQL ウェアハウスに接続）。

In [0]:
dbutils.widgets.text("catalog", "fevm_jp_hunter_serverless_catalog", "カタログ名")
dbutils.widgets.text("schema", "bricks_electronics", "スキーマ名")
catalog = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema_name}`")
print(f"{catalog}.{schema_name} を使います")

In [0]:
display(spark.sql("""
WITH s AS (
  SELECT model_no, sales_month, SUM(units_sold) AS units FROM sales_monthly GROUP BY model_no, sales_month
), r AS (
  SELECT model_no, DATE_TRUNC('MONTH', received_at) AS m, COUNT(*) AS n FROM repair_requests GROUP BY model_no, DATE_TRUNC('MONTH', received_at)
), j AS (
  SELECT s.model_no, s.sales_month, s.units, COALESCE(r.n, 0) AS n,
         CASE WHEN s.sales_month BETWEEN '2025-10-01' AND '2026-03-01' THEN '前の半年'
              WHEN s.sales_month BETWEEN '2026-04-01' AND '2026-09-01' THEN 'この半年' END AS period
  FROM s LEFT JOIN r ON s.model_no = r.model_no AND s.sales_month = r.m
)
SELECT model_no,
       SUM(IF(period = '前の半年', n, 0)) AS `前の半年_件数`,
       ROUND(SUM(IF(period = '前の半年', n, 0)) / SUM(IF(period = '前の半年', units, 0)) * 1000, 1) AS `前の半年_受付率`,
       SUM(IF(period = 'この半年', n, 0)) AS `この半年_件数`,
       ROUND(SUM(IF(period = 'この半年', n, 0)) / SUM(IF(period = 'この半年', units, 0)) * 1000, 1) AS `この半年_受付率`
FROM j WHERE period IS NOT NULL
GROUP BY model_no ORDER BY `この半年_受付率` / `前の半年_受付率` DESC
"""))

In [0]:
display(spark.sql("""
SELECT l.lot_no, l.factory, l.production_month, l.quantity,
       COUNT(r.request_id) AS `受付件数`,
       SUM(IF(r.symptom_category = '途中で止まる', 1, 0)) AS `途中で止まる`,
       ROUND(COUNT(r.request_id) / l.quantity * 1000, 1) AS `製造1000台あたり`
FROM production_lots l LEFT JOIN repair_requests r ON l.lot_no = r.lot_no
WHERE l.model_no = 'RC-500' AND l.production_month >= '2026-01-01'
GROUP BY l.lot_no, l.factory, l.production_month, l.quantity
ORDER BY l.production_month, l.lot_no
"""))

In [0]:
display(spark.sql("""
SELECT COUNT(DISTINCT l.lot_no) AS `ロット数`,
       (SELECT SUM(quantity) FROM production_lots WHERE model_no = 'RC-500' AND factory = 'タイ工場'
          AND production_month BETWEEN '2026-04-01' AND '2026-06-01') AS `対象台数`,
       COUNT(r.request_id) AS `これまでの受付`,
       SUM(r.service_cost_jpy) AS `当社負担_円`,
       ROUND(AVG(IF(r.result_category = '持込修理', r.service_cost_jpy, NULL))) AS `持込修理の平均_円`
FROM production_lots l JOIN repair_requests r ON l.lot_no = r.lot_no
WHERE l.model_no = 'RC-500' AND l.factory = 'タイ工場' AND l.production_month BETWEEN '2026-04-01' AND '2026-06-01'
"""))

In [0]:
display(spark.sql("""
SELECT channel AS `チャネル`, COUNT(*) AS `受付`,
       SUM(IF(result_category = '修理不要', 1, 0)) AS `修理不要`,
       SUM(IF(result_category = '異常なし', 1, 0)) AS `異常なし`,
       ROUND(AVG(IF(result_category = '異常なし', 1, 0)) * 100, 1) AS `異常なし率_pct`,
       SUM(IF(result_category = '異常なし', service_cost_jpy, 0)) AS `異常なしの当社負担_円`
FROM repair_requests GROUP BY channel
"""))

In [0]:
display(spark.sql("""
SELECT model_no, symptom_category, COUNT(*) AS `受付`,
       SUM(IF(result_category IN ('修理不要', '異常なし'), 1, 0)) AS `修理が不要だった`,
       ROUND(AVG(IF(result_category IN ('修理不要', '異常なし'), 1, 0)) * 100) AS `割合_pct`
FROM repair_requests
GROUP BY model_no, symptom_category
HAVING COUNT(*) >= 100
ORDER BY `修理が不要だった` DESC
LIMIT 12
"""))